# Citi Bike Demand Imbalance EDA — Data Acquisition Framework

**Author:** Skye Xi\
**Course:** Data Bootcamp\
**Date:** Spring 2026

---

**This notebook handles:**
1. Downloading Citi Bike trip data (multiple months)
2. Fetching NYC weather data from NOAA API
3. Merging datasets into a clean, analysis-ready DataFrame
4. Saving outputs to `/data/` for use in later notebooks

> **Reproducibility note:** All raw files are saved locally so the pipeline can be re-run without re-downloading. Set `FORCE_DOWNLOAD = True` to refresh.

In [10]:
import gc
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path


## Section 1: Data Collection & Acquisition

This cell handles:

  1a. Configuration\
  1b. Load raw trip data from extracted folders\
  1c. Clean trip data & engineer features\
  1d. NOAA weather data via API\
  1e. Build analysis-ready dataFrames\
  1f. Save processed outputs to /data/processed/

> **Reproducibility note:**\
  Raw ZIPs are cached locally after first download.\
  Set FORCE_DOWNLOAD = True to refresh everything.

In [11]:
# ============================================================
# 1a. CONFIGURATION
# ============================================================

# ── NOAA Weather API ─────────────────────────────────────────
# Free token: https://www.ncdc.noaa.gov/cdo-web/token
NOAA_TOKEN      = "WGgSWrziydfZmtQDMDsxozetCluUNYAs"
NOAA_STATION_ID = "GHCND:USW00094728"    # NYC Central Park

# ── Months to analyze ────────────────────────────────────────
MONTHS = [
    "2025-03", "2025-04", "2025-05",
    "2025-06", "2025-07", "2025-08",
    "2025-09", "2025-10", "2025-11",
    "2025-12", "2026-01", "2026-02",
]

# ── Data directories ─────────────────────────────────────────
DATA_DIR = Path("/Users/shrinn81/Library/CloudStorage/"
                "GoogleDrive-skye.xry@gmail.com/"
                "我的云端硬盘/Colab Notebooks/Data Bootcamp/citi_bike/data")

PROC_DIR = DATA_DIR / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("\u2713 Configuration loaded.")
print(f"  Months    : {MONTHS[0]} → {MONTHS[-1]}  ({len(MONTHS)} months)")
print(f"  Data dir  : {DATA_DIR}")
print(f"  Proc dir  : {PROC_DIR}")
print(f"  Data dir exists : {DATA_DIR.exists()}")


✓ Configuration loaded.
  Months    : 2025-03 → 2026-02  (12 months)
  Data dir  : /Users/shrinn81/Library/CloudStorage/GoogleDrive-skye.xry@gmail.com/我的云端硬盘/Colab Notebooks/Data Bootcamp/citi_bike/data
  Proc dir  : /Users/shrinn81/Library/CloudStorage/GoogleDrive-skye.xry@gmail.com/我的云端硬盘/Colab Notebooks/Data Bootcamp/citi_bike/data/processed
  Data dir exists : True


In [12]:
# ============================================================
# 1b. LOAD RAW TRIP DATA FROM EXTRACTED FOLDERS
# ============================================================
#
# Expected folder structure inside DATA_DIR:
#   202509-citibike-tripdata/
#     202509-citibike-tripdata_1.csv
#     202509-citibike-tripdata_2.csv   ← some months are split
#   202510-citibike-tripdata/
#     202510-citibike-tripdata.csv
#   ...
# ============================================================
 
frames = []
failed = []
 
print(f"\n[1b] Loading {len(MONTHS)} months from local folders...\n")
 
for ym in MONTHS:
    year, month = ym.split("-")
    folder_name = f"{year}{month}-citibike-tripdata"
    folder_path = DATA_DIR / folder_name
 
    if not folder_path.exists():
        print(f"  ✗ Folder not found : {folder_path}")
        failed.append(ym)
        continue
 
    csv_files = sorted(folder_path.glob("*.csv"))
    if not csv_files:
        print(f"  ✗ No CSV files in  : {folder_path}")
        failed.append(ym)
        continue
 
    print(f"  Reading : {folder_name}/  ({len(csv_files)} file(s))")
    parts = []
    for csv_path in csv_files:
        df_part = pd.read_csv(csv_path, low_memory=False)
        parts.append(df_part)
        print(f"    {csv_path.name}  →  {len(df_part):,} rows")
 
    df_month = pd.concat(parts, ignore_index=True)
    df_month["source_month"] = ym
    frames.append(df_month)
    print(f"  ✓ {ym} subtotal : {len(df_month):,} rows\n")
 
if failed:
    print(f"⚠ Months not loaded : {failed}")
if not frames:
    raise RuntimeError("No data loaded. Check DATA_DIR and folder names.")
 
df_trips_raw = pd.concat(frames, ignore_index=True)
 
print(f"✓ Raw trips loaded : {len(df_trips_raw):,} rows")
print(f"  Columns          : {list(df_trips_raw.columns)}")


[1b] Loading 12 months from local folders...

  Reading : 202503-citibike-tripdata/  (4 file(s))
    202503-citibike-tripdata_1.csv  →  1,000,000 rows
    202503-citibike-tripdata_2.csv  →  1,000,000 rows
    202503-citibike-tripdata_3.csv  →  1,000,000 rows
    202503-citibike-tripdata_4.csv  →  168,271 rows
  ✓ 2025-03 subtotal : 3,168,271 rows

  Reading : 202504-citibike-tripdata/  (4 file(s))
    202504-citibike-tripdata_1.csv  →  1,000,000 rows
    202504-citibike-tripdata_2.csv  →  1,000,000 rows
    202504-citibike-tripdata_3.csv  →  1,000,000 rows
    202504-citibike-tripdata_4.csv  →  724,596 rows
  ✓ 2025-04 subtotal : 3,724,596 rows

  Reading : 202505-citibike-tripdata/  (5 file(s))
    202505-citibike-tripdata_1.csv  →  1,000,000 rows
    202505-citibike-tripdata_2.csv  →  1,000,000 rows
    202505-citibike-tripdata_3.csv  →  1,000,000 rows
    202505-citibike-tripdata_4.csv  →  1,000,000 rows
    202505-citibike-tripdata_5.csv  →  325,553 rows
  ✓ 2025-05 subtotal : 4,3

In [13]:
# ============================================================
# 1c. CLEAN CITI BIKE TRIP DATA & ENGINEER FEATURES
# ============================================================
#
# Steps:
#   i.   Parse datetime columns
#   ii.  Drop rows missing critical fields
#   iii. Remove implausible trip durations
#   iv.  Standardize user_type column
#   v.   Temporal features: date, hour, day_of_week, day_name,
#                           month, year, season, is_weekend, rush_hour
# ============================================================

print("\n[1c] Cleaning Citi Bike trip data...")

df = df_trips_raw.copy()

# ── i. Parse datetimes ────────────────────────────────────────
df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
df["ended_at"]   = pd.to_datetime(df["ended_at"],   errors="coerce")

# ── ii. Drop rows missing critical fields ─────────────────────
n0 = len(df)
df.dropna(subset=["started_at", "ended_at",
                   "start_station_id", "end_station_id",
                   "start_lat", "start_lng"], inplace=True)
print(f"  Dropped {n0 - len(df):,} rows  — null datetime / station / coords")

# ── iii. Trip duration filter ─────────────────────────────────
df["duration_min"] = (
    (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
)
n1 = len(df)
df = df[df["duration_min"].between(1, 180)].copy()
print(f"  Dropped {n1 - len(df):,} rows  — duration <1 min or >180 min")
print(f"  ✓ Clean rows remaining : {len(df):,}")

# ── iv. Standardize user_type ─────────────────────────────────
if "member_casual" in df.columns:
    df["user_type"] = df["member_casual"]
elif "usertype" in df.columns:
    df["user_type"] = (
        df["usertype"]
        .str.lower()
        .str.replace("subscriber", "member", regex=False)
        .str.replace("customer",   "casual",  regex=False)
    )
else:
    df["user_type"] = "unknown"
    print("  ⚠ No user type column found — set to 'unknown'")

# ── v. Temporal feature engineering ───────────────────────────
df["date"]        = df["started_at"].dt.date
df["hour"]        = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.dayofweek   # 0=Mon, 6=Sun
df["day_name"]    = df["started_at"].dt.day_name()  # 'Monday', 'Tuesday', ...
df["month"]       = df["started_at"].dt.month
df["year"]        = df["started_at"].dt.year
df["is_weekend"]  = df["day_of_week"].isin([5, 6]).astype(int)

# Season (meteorological)
df["season"] = df["month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring",  4: "Spring", 5: "Spring",
    6: "Summer",  7: "Summer", 8: "Summer",
    9: "Fall",   10: "Fall",  11: "Fall",
})

# Rush hour: weekday 7–9 AM or 5–7 PM
df["rush_hour"] = (
    (df["is_weekend"] == 0) &
    (df["hour"].isin([7, 8, 17, 18]))
).astype(int)

# Rename cleaned DataFrame
df_trips = df.copy()
del df, df_trips_raw
gc.collect()

# ── Summary ───────────────────────────────────────────────────
print(f"\n  Date range   : {df_trips['date'].min()} → {df_trips['date'].max()}")
print(f"  user_type    :\n{df_trips['user_type'].value_counts().to_string()}")
print(f"  season       :\n{df_trips['season'].value_counts().to_string()}")
print(f"  rush_hour    : {df_trips['rush_hour'].sum():,} trips ({df_trips['rush_hour'].mean():.1%})")
print(f"  duration_min : mean={df_trips['duration_min'].mean():.1f}  "
      f"median={df_trips['duration_min'].median():.1f}  "
      f"max={df_trips['duration_min'].max():.1f}")
print(f"\n  ✓ df_trips shape : {df_trips.shape}")
print(f"  Columns          : {list(df_trips.columns)}")



[1c] Cleaning Citi Bike trip data...
  Dropped 156,064 rows  — null datetime / station / coords
  Dropped 32,085 rows  — duration <1 min or >180 min
  ✓ Clean rows remaining : 44,463,056

  Date range   : 2025-02-28 → 2026-02-28
  user_type    :
user_type
member    36674240
casual     7788816
  season       :
season
Summer    14799003
Fall      13387229
Spring    11177203
Winter     5099621
  rush_hour    : 10,394,596 trips (23.4%)
  duration_min : mean=12.2  median=8.9  max=180.0

  ✓ df_trips shape : (44463056, 25)
  Columns          : ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'source_month', 'duration_min', 'user_type', 'date', 'hour', 'day_of_week', 'day_name', 'month', 'year', 'is_weekend', 'season', 'rush_hour']


In [14]:
# ============================================================
# 1d. NOAA WEATHER DATA (API)
# ============================================================
#
# Endpoint  : https://www.ncdc.noaa.gov/cdo-web/api/v2/data
# Dataset   : GHCND (Global Historical Climatology Network — Daily)
# Station   : NYC Central Park (GHCND:USW00094728)
#
# Features fetched:
#   TMAX — max daily temp      (°C)
#   TMIN — min daily temp      (°C)
#   PRCP — precipitation       (mm)
#   SNOW — snowfall            (mm)
#   AWND — avg wind speed      (m/s)
#
# ⚠ API LIMIT: The NOAA CDO API returns at most 1,000 records per
#   request.  With 5 datatypes × 365 days = 1,825 records needed,
#   a single call silently truncates at ~200 days.  We fix this by
#   fetching one calendar month at a time (5 × 31 = 155 records max)
#   and concatenating the results.
# ============================================================

import time

def fetch_noaa_month(year_month: str, token: str, station_id: str) -> pd.DataFrame:
    """
    Fetch NOAA daily weather for a single calendar month.

    Fetching month-by-month stays well within the 1,000-record API limit
    (5 datatypes × 31 days = 155 records max per call).

    Args:
        year_month : "YYYY-MM"
        token      : NOAA CDO API token
        station_id : e.g. "GHCND:USW00094728"

    Returns:
        Wide-format DataFrame (one row per day) or empty DataFrame on error.
    """
    import calendar
    year, month = int(year_month[:4]), int(year_month[5:7])
    last_day = calendar.monthrange(year, month)[1]
    start_date = f"{year_month}-01"
    end_date   = f"{year_month}-{last_day:02d}"

    headers   = {"token": token}
    datatypes = ["TMAX", "TMIN", "PRCP", "SNOW", "AWND"]
    params    = {
        "datasetid":  "GHCND",
        "stationid":  station_id,
        "startdate":  start_date,
        "enddate":    end_date,
        "datatypeid": ",".join(datatypes),
        "limit":      1000,        # 155 records max per month — well within limit
        "units":      "metric",
    }

    try:
        r = requests.get(
            "https://www.ncdc.noaa.gov/cdo-web/api/v2/data",
            headers=headers, params=params, timeout=30
        )
    except requests.RequestException as e:
        print(f"    ✗ {year_month}: request failed — {e}")
        return pd.DataFrame()

    if r.status_code != 200:
        print(f"    ✗ {year_month}: API error {r.status_code} — {r.text[:120]}")
        return pd.DataFrame()

    data = r.json()
    if "results" not in data or not data["results"]:
        print(f"    ⚠ {year_month}: no results returned")
        return pd.DataFrame()

    # Pivot: long → wide (one row per day)
    df_w = (
        pd.DataFrame(data["results"])
        .assign(date=lambda x: pd.to_datetime(x["date"]).dt.date)
        .pivot_table(index="date", columns="datatype",
                     values="value", aggfunc="first")
        .reset_index()
    )
    df_w.columns.name = None
    return df_w


def fetch_noaa_daily(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetch NOAA daily weather for an arbitrary date range.

    Splits the range into monthly chunks to stay within the API's
    1,000-record-per-request limit, then concatenates and post-processes.

    Args:
        start_date : "YYYY-MM-DD"
        end_date   : "YYYY-MM-DD"

    Returns:
        Wide-format DataFrame (one row per day) with derived columns:
        TAVG, is_rainy, is_snowy, is_bad_weather.
        Returns empty DataFrame if token is missing or all calls fail.
    """
    if not NOAA_TOKEN:
        print("  ⚠ NOAA_TOKEN not set — skipping weather fetch.")
        return pd.DataFrame()

    # Build list of "YYYY-MM" strings spanning the requested range
    start_ym = pd.Period(start_date[:7], freq="M")
    end_ym   = pd.Period(end_date[:7],   freq="M")
    months   = [str(p) for p in pd.period_range(start_ym, end_ym, freq="M")]

    print(f"  Fetching NOAA data month-by-month ({len(months)} requests)...")

    chunks = []
    for ym in months:
        df_chunk = fetch_noaa_month(ym, NOAA_TOKEN, NOAA_STATION_ID)
        if not df_chunk.empty:
            print(f"    ✓ {ym}: {len(df_chunk)} days")
            chunks.append(df_chunk)
        else:
            print(f"    ✗ {ym}: no data")
        time.sleep(0.3)   # be polite to the API (rate limit: ~5 req/s)

    if not chunks:
        print("  ⚠ No weather data retrieved.")
        return pd.DataFrame()

    df_weather = pd.concat(chunks, ignore_index=True)

    # Keep only rows within the requested date window
    df_weather["date"] = pd.to_datetime(df_weather["date"])
    mask = (df_weather["date"] >= start_date) & (df_weather["date"] <= end_date)
    df_weather = df_weather[mask].reset_index(drop=True)

    # Derived columns
    if {"TMAX", "TMIN"}.issubset(df_weather.columns):
        df_weather["TAVG"] = (df_weather["TMAX"] + df_weather["TMIN"]) / 2

    df_weather["is_rainy"]       = (df_weather.get("PRCP", pd.Series(0)) > 1.0).astype(float)
    df_weather["is_snowy"]       = (df_weather.get("SNOW", pd.Series(0)) > 0.0).astype(float)
    df_weather["is_bad_weather"] = ((df_weather["is_rainy"] == 1) | (df_weather["is_snowy"] == 1)).astype(float)

    coverage = len(df_weather)
    expected = (pd.to_datetime(end_date) - pd.to_datetime(start_date)).days + 1
    print(f"  ✓ Weather fetch complete: {coverage}/{expected} days ({coverage/expected:.0%} coverage)")
    return df_weather


print("\n[1d] Fetching NOAA weather data...")
min_date   = str(df_trips["date"].min())
max_date   = str(df_trips["date"].max())
df_weather = fetch_noaa_daily(min_date, max_date)

if df_weather.empty:
    print("  ⚠ Proceeding without weather data.")
else:
    print(f"  ✓ df_weather shape : {df_weather.shape}")
    print(f"  Columns : {list(df_weather.columns)}")
    print(df_weather.head(3).to_string(index=False))



[1d] Fetching NOAA weather data...
  Fetching NOAA data month-by-month (13 requests)...
    ✓ 2025-02: 28 days
    ✓ 2025-03: 31 days
    ✓ 2025-04: 30 days
    ✓ 2025-05: 31 days
    ✓ 2025-06: 30 days
    ✓ 2025-07: 31 days
    ✓ 2025-08: 31 days
    ✓ 2025-09: 30 days
    ✓ 2025-10: 31 days
    ✓ 2025-11: 30 days
    ✓ 2025-12: 31 days
    ✓ 2026-01: 31 days
    ✓ 2026-02: 28 days
  ✓ Weather fetch complete: 366/366 days (100% coverage)
  ✓ df_weather shape : (366, 10)
  Columns : ['date', 'AWND', 'PRCP', 'SNOW', 'TMAX', 'TMIN', 'TAVG', 'is_rainy', 'is_snowy', 'is_bad_weather']
      date  AWND  PRCP  SNOW  TMAX  TMIN  TAVG  is_rainy  is_snowy  is_bad_weather
2025-02-28   3.0   0.0   0.0  10.6   5.0  7.80       0.0       0.0             0.0
2025-03-01   4.3   0.0   0.0  17.8  -2.7  7.55       0.0       0.0             0.0
2025-03-02   4.3   0.0   0.0   0.6  -6.6 -3.00       0.0       0.0             0.0


In [15]:
# ============================================================
# 1e. BUILD ANALYSIS-READY DATAFRAMES
# ============================================================
#
# Three tables derived from df_trips:
#
#   df_daily      : one row per day — system-wide stats + weather
#                   → time series, weather correlation, rideable mix
#
#   df_station    : one row per station — departures, arrivals,
#                   net_flow, coords → map, imbalance ranking
#
#   trips_sample  : 100k random trips — row-level detail
#                   → hourly distribution, duration histogram,
#                     user type / ride type breakdown
#
# df_trips is released from memory after aggregation.
# ============================================================

print("\n[1e] Building analysis-ready DataFrames...")

# ── df_daily : system-wide daily aggregation ─────────────────
df_daily = (
    df_trips.groupby("date")
    .agg(
        total_rides     = ("ride_id",          "count"),
        member_rides    = ("user_type",         lambda x: (x == "member").sum()),
        casual_rides    = ("user_type",         lambda x: (x == "casual").sum()),
        electric_rides  = ("rideable_type",     lambda x: x.str.contains("electric", case=False, na=False).sum()),
        classic_rides   = ("rideable_type",     lambda x: x.str.contains("classic",  case=False, na=False).sum()),
        rush_hour_rides = ("rush_hour",         "sum"),
        avg_duration    = ("duration_min",      "mean"),
        median_duration = ("duration_min",      "median"),
        unique_stations = ("start_station_id",  "nunique"),
    )
    .reset_index()
)

df_daily["pct_member"]      = (df_daily["member_rides"]   / df_daily["total_rides"] * 100).round(2)
df_daily["pct_casual"]      = (df_daily["casual_rides"]   / df_daily["total_rides"] * 100).round(2)
df_daily["pct_electric"]    = (df_daily["electric_rides"] / df_daily["total_rides"] * 100).round(2)
df_daily["pct_rush_hour"]   = (df_daily["rush_hour_rides"]/ df_daily["total_rides"] * 100).round(2)
df_daily["avg_duration"]    = df_daily["avg_duration"].round(2)
df_daily["median_duration"] = df_daily["median_duration"].round(2)

# Temporal features
df_daily["date"]        = pd.to_datetime(df_daily["date"])
df_daily["day_of_week"] = df_daily["date"].dt.dayofweek
df_daily["day_name"]    = df_daily["date"].dt.day_name()
df_daily["month"]       = df_daily["date"].dt.month
df_daily["year"]        = df_daily["date"].dt.year
df_daily["is_weekend"]  = df_daily["day_of_week"].isin([5, 6]).astype(int)
df_daily["season"]      = df_daily["month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring",  4: "Spring", 5: "Spring",
    6: "Summer",  7: "Summer", 8: "Summer",
    9: "Fall",   10: "Fall",  11: "Fall",
})

# Merge weather
if not df_weather.empty:
    df_weather["date"] = pd.to_datetime(df_weather["date"])
    df_daily = pd.merge(df_daily, df_weather, on="date", how="left")
    coverage = df_daily["TAVG"].notna().mean()
    print(f"  ✓ df_daily shape         : {df_daily.shape}")
    print(f"    Weather join coverage  : {coverage:.1%}")
else:
    print(f"  ✓ df_daily shape         : {df_daily.shape}  (no weather joined)")

df_daily = df_daily.sort_values("date").reset_index(drop=True)


# ── df_station : per-station departures + arrivals + net_flow ─
dep = (
    df_trips.groupby(["start_station_id", "start_station_name"])
    .agg(
        total_departures = ("ride_id",       "count"),
        member_dep       = ("user_type",      lambda x: (x == "member").sum()),
        casual_dep       = ("user_type",      lambda x: (x == "casual").sum()),
        electric_dep     = ("rideable_type",  lambda x: x.str.contains("electric", case=False, na=False).sum()),
        lat              = ("start_lat",      "median"),
        lng              = ("start_lng",      "median"),
        avg_duration     = ("duration_min",   "mean"),
    )
    .reset_index()
    .rename(columns={"start_station_id": "station_id", "start_station_name": "station_name"})
)

arr = (
    df_trips.groupby("end_station_id")
    .size()
    .reset_index(name="total_arrivals")
    .rename(columns={"end_station_id": "station_id"})
)

df_station = pd.merge(dep, arr, on="station_id", how="left").fillna(0)
df_station["net_flow"]   = df_station["total_arrivals"] - df_station["total_departures"]
df_station["pct_member"] = (df_station["member_dep"] / df_station["total_departures"] * 100).round(2)
df_station["avg_duration"] = df_station["avg_duration"].round(2)
df_station = df_station.sort_values("total_departures", ascending=False).reset_index(drop=True)

print(f"  ✓ df_station shape       : {df_station.shape}")
print(f"    Top station            : {df_station.iloc[0]['station_name']}  "
      f"({df_station.iloc[0]['total_departures']:,.0f} dep)")
net_importer = df_station.loc[df_station['net_flow'].idxmax(), 'station_name']
net_exporter = df_station.loc[df_station['net_flow'].idxmin(), 'station_name']
print(f"    Biggest importer       : {net_importer}  ({df_station['net_flow'].max():+,.0f})")
print(f"    Biggest exporter       : {net_exporter}  ({df_station['net_flow'].min():+,.0f})")


# ── trips_sample : 100k random row-level trips ───────────────
SAMPLE_COLS = [
    "ride_id", "rideable_type",
    "started_at", "ended_at", "duration_min",
    "start_station_id", "start_station_name",
    "end_station_id",   "end_station_name",
    "start_lat", "start_lng", "end_lat", "end_lng",
    "user_type",
    "date", "hour", "day_of_week", "day_name",
    "month", "year", "season", "is_weekend", "rush_hour",
    "source_month",
]
available_cols = [c for c in SAMPLE_COLS if c in df_trips.columns]

trips_sample = (
    df_trips[available_cols]
    .sample(min(100_000, len(df_trips)), random_state=42)
    .reset_index(drop=True)
)
print(f"  ✓ trips_sample shape     : {trips_sample.shape}")


# ── Release raw trips from memory ────────────────────────────
del df_trips
gc.collect()
print(f"\n  ✓ df_trips released from memory")


# ============================================================
# 1f. SAVE PROCESSED DATA
# ============================================================

print("\n[1f] Saving processed files...")

df_daily_export = df_daily.copy()
df_daily_export["date"] = df_daily_export["date"].dt.strftime("%Y-%m-%d")

path_daily   = PROC_DIR / "daily_rides_weather.csv"
path_station = PROC_DIR / "station_summary.csv"
path_sample  = PROC_DIR / "trips_sample.csv"

df_daily_export.to_csv(path_daily,   index=False)
df_station.to_csv(path_station,      index=False)
trips_sample.to_csv(path_sample,     index=False)

print(f"  ✓ daily_rides_weather.csv  —  {len(df_daily_export)} rows  ×  {df_daily_export.shape[1]} cols")
print(f"  ✓ station_summary.csv      —  {len(df_station)} rows  ×  {df_station.shape[1]} cols")
print(f"  ✓ trips_sample.csv         —  {len(trips_sample)} rows  ×  {trips_sample.shape[1]} cols")
print(f"\n  Saved to: {PROC_DIR.resolve()}")


# ============================================================
# SECTION 1 SUMMARY
# ============================================================

print("\n" + "=" * 62)
print("  SECTION 1 COMPLETE")
print("=" * 62)
print(f"  Months loaded      : {MONTHS[0]} → {MONTHS[-1]}")
print(f"  Date range         : {df_daily['date'].min().date()} → {df_daily['date'].max().date()}")
print(f"  Total clean trips  : {df_daily['total_rides'].sum():,}")
print(f"  Unique stations    : {len(df_station)}")
print(f"  Weather joined     : {'Yes' if not df_weather.empty else 'No'}")
print(f"\n  DataFrames ready   :")
print(f"    df_daily          {df_daily.shape}")
print(f"    df_station        {df_station.shape}")
print(f"    trips_sample      {trips_sample.shape}")
print(f"\n  Key columns in df_daily:")
print(f"    rides, user mix, electric mix, rush hour, duration,")
print(f"    day_name, season, is_weekend, TMAX/TMIN/TAVG/PRCP/SNOW/AWND")
print(f"\n  Key columns in df_station:")
print(f"    departures, arrivals, net_flow, lat/lng, pct_member")
print(f"\n  ► Next: Section 2 — EDA")
print("=" * 62)



[1e] Building analysis-ready DataFrames...
  ✓ df_daily shape         : (365, 29)
    Weather join coverage  : 100.0%
  ✓ df_station shape       : (2352, 12)
    Top station            : W 21 St & 6 Ave  (161,668 dep)
    Biggest importer       : E 17 St & Broadway  (+37,209)
    Biggest exporter       : E 17 St & Broadway  (-36,245)
  ✓ trips_sample shape     : (100000, 24)

  ✓ df_trips released from memory

[1f] Saving processed files...
  ✓ daily_rides_weather.csv  —  365 rows  ×  29 cols
  ✓ station_summary.csv      —  2352 rows  ×  12 cols
  ✓ trips_sample.csv         —  100000 rows  ×  24 cols

  Saved to: /Users/shrinn81/Library/CloudStorage/GoogleDrive-skye.xry@gmail.com/我的云端硬盘/Colab Notebooks/Data Bootcamp/citi_bike/data/processed

  SECTION 1 COMPLETE
  Months loaded      : 2025-03 → 2026-02
  Date range         : 2025-02-28 → 2026-02-28
  Total clean trips  : 44,463,056
  Unique stations    : 2352
  Weather joined     : Yes

  DataFrames ready   :
    df_daily          (36